# FCA–MoE F2-E — One-shot confirmatory campaign

This notebook runs the temporal leave-one-sequence-out prerequisite first. Only if it passes does the campaign runner launch the full nested parametric bootstrap. It uses the existing Qwen1.5-MoE-A2.7B layer-0 traces (four sequences × 256 tokens, 60 routed experts, exact Top-4). The full run is computationally substantial and checkpoints every outer replicate.

**This notebook does not claim an empirical result until the Colab/Drive campaign has completed and its complete output package has been audited.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess
REPO = Path('/content/FCA-MoE')
BRANCH = 'research/fca-moe-v0.3-f2e-nested-bootstrap'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/msellamiTN/FCA-MoE.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
%cd /content/FCA-MoE/research/v03
!python -m pip install -e .
!python -m pip install pytest


In [ ]:
# Full v0.3 test suite, including transition-kernel normalization, invariance, detailed balance and exact transition likelihood.
!python -m pytest -q


In [ ]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/FCA_MoE_v03/qwen15_moe_a27_wikitext_test/contexts_v1')
CAMPAIGN_OUT = ROOT/'sequence_replication/layer_00/temporal/f2e_confirmatory_campaign_v1'
required = [ROOT/'qwen_contexts_by_layer.npz', ROOT/'sequence_alignment.csv']
for path in required:
    print(path, path.exists())
    if not path.exists(): raise FileNotFoundError(path)
print('Campaign output:', CAMPAIGN_OUT)


## Final one-shot execution

The runner performs four-fold leave-one-sequence-out temporal validation using training-only lambda and weight selection. It evaluates the exact conditional transition likelihood on each held-out sequence. The predeclared diagnostic must show a positive mean transition-NLL gain over IID and improvement in at least three of four folds. If it fails, the campaign stops before the expensive bootstrap.

If the LOO gate passes, the runner launches 999 outer × 100 inner replicates, repeats lambda and temporal-weight selection inside every outer replicate, preserves all six FCA metrics, and writes full audit artifacts and checkpoints.


In [ ]:
import subprocess, sys

# First execution: False. After an interruption, set True and rerun this same cell.
RESUME = False

cmd = [
    sys.executable, 'scripts/run_f2e_confirmatory_campaign.py',
    '--root', str(ROOT),
    '--output-base', str(CAMPAIGN_OUT),
    '--outer-replicates', '999',
    '--inner-replicates', '100',
    '--screen-replicates', '8',
    '--weight-grid-step', '0.1',
    '--lambdas', '0.0003', '0.001', '0.003',
    '--maxiter', '400',
    '--predictive-replicates', '200',
    '--seed', '20261120',
]
if RESUME:
    cmd.append('--resume')

print('Launching:', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
import json
campaign_manifest = CAMPAIGN_OUT/'campaign_manifest.json'
bootstrap_summary = CAMPAIGN_OUT/'nested_bootstrap/f2e_nested_bootstrap_summary.json'
if campaign_manifest.exists():
    print('CAMPAIGN MANIFEST')
    print(json.dumps(json.loads(campaign_manifest.read_text()), indent=2))
if bootstrap_summary.exists():
    print('NESTED BOOTSTRAP SUMMARY')
    print(json.dumps(json.loads(bootstrap_summary.read_text()), indent=2))
else:
    print('Nested bootstrap did not run or has not completed. Inspect campaign_manifest.json and temporal_loo outputs.')


## Output package

The campaign output contains the LOO fold selections, full weight screens, held-out transition likelihoods, lag-profile predictive results, frozen code/data hashes, observed inner-null values, all outer weight screens, all outer inner-null metrics, fitted parameter artifacts, checkpoint CSV, and final nested-bootstrap summary.

After completion, archive the entire `f2e_confirmatory_campaign_v1` directory. Do not send only the summary JSON: the individual null draws and model-selection records are required for independent audit.


## Interpretation guardrails

- The LOO diagnostic is descriptive with four sequences; it is not a significance test or proof of broad generalization.
- The nested inference is conditional on the specified pairwise-temporal null family and does not protect against arbitrary misspecification.
- A statistically significant result would support higher-order FCA excess beyond this fitted null for this model, dataset, sample and layer; it would not establish semantic specialization, causality, interpretability gains, or inference-speed gains.
- Four source sequences and layer 0 do not justify corpus-wide or model-wide claims.
